# Week 1 · Day 4 — Lab 1
## Acquiring Data over HTTP with `requests`

> **AI Engineering Academy** · Gamut Technology Services

Every data pipeline begins by pulling data from somewhere — usually a REST API.
Doing it *reliably* means a specific set of non-negotiable habits: always set a
**timeout**, always call **`raise_for_status()`**, reuse a **`Session`**, configure
**retries with backoff**, handle **pagination** explicitly, respect **rate limits**,
and read **secrets from the environment**. This lab drills all of them.

### ⚙️ You are calling a LOCAL API — no internet required
You cannot reach external APIs from this environment, and you shouldn't need to.
The setup cell starts a small **FastAPI** server (`lab_api.py`, shipped beside this
notebook) on `http://127.0.0.1:8001`. It behaves like a real API — bearer-token
auth, pagination, transient failures, 429s, and a POST endpoint — so every habit
below is practiced against something realistic. See **`API_REFERENCE.md`** for the
full endpoint list, inputs, and outputs.

### Learning objectives
1. Make a `GET` with params, headers, and a **timeout**, and fail loudly with `raise_for_status()`.
2. Reuse a **`requests.Session`** with shared auth and connection pooling.
3. Assemble a full dataset across **page/per_page** and **cursor** pagination.
4. Recover from transient failures with **`Retry` + `HTTPAdapter`** (exponential backoff).
5. Respect a **`Retry-After`** header on `429` responses.
6. Send a `POST` with a **JSON body** using `json=`.
7. Keep **secrets in environment variables**, never in source.

### Time budget — ~88 min
| Segment | Time |
|---|---|
| Setup & the local API | 6 min |
| **A.** First GET, status codes, `raise_for_status`, timeout | 14 min |
| **B.** Sessions | 10 min |
| **C.** Pagination (page / per_page) | 14 min |
| **D.** Cursor pagination | 12 min |
| **E.** Retries & exponential backoff | 14 min |
| **F.** Rate limits (`Retry-After`) | 8 min |
| **G.** POST with a JSON body | 8 min |
| Wrap-up + stretch | 2 min |

### Files you need (beside this notebook)
- `lab_api.py` — the local practice API (started for you in the setup cell).
- `API_REFERENCE.md` — endpoint documentation.
- `.env.example` — the environment-variable template (for real projects).


In [ ]:
%pip install --upgrade pandas requests fastapi uvicorn pydantic

In [ ]:
# --- Setup: start the local practice API and configure the client ---------
# The student CANNOT call external APIs. Instead, `lab_api.py` (shipped beside
# this notebook) stands up a small FastAPI server on localhost that behaves like
# a real REST API: pagination, retries, rate limits, auth, and POST endpoints.
import os, time, requests
import pandas as pd
from lab_api import start_server

# Secrets come from the ENVIRONMENT, never hardcoded in code (deck: slides 17-18).
# In a real project these would live in a .env file loaded by python-dotenv, or a
# secrets manager. Here we seed the known local-dev value if it is not already set.
os.environ.setdefault("LAB_API_KEY", "local-dev-key")   # the server expects this
os.environ.setdefault("API_KEY", os.environ["LAB_API_KEY"])  # the client reads this

BASE_URL = "http://127.0.0.1:8001"

# Launch the API on a background thread and wait until it answers /health.
server, _thread = start_server(port=8001)
for _ in range(50):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.1)
print("Local API ready at", BASE_URL)

def auth_headers() -> dict:
    """Build auth headers by reading the key from the environment (never inline)."""
    return {"Authorization": f"Bearer {os.environ['API_KEY']}", "Accept": "application/json"}

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — Your first GET: status codes, `raise_for_status`, timeout  *(guided)*

Four habits live in one call: pass **`params`** for the query string, **`headers`**
for auth, a **`timeout`** so a hung server can't block forever, and
**`raise_for_status()`** to turn 4xx/5xx into a loud Python exception instead of a
silent bad dataset.


In [ ]:
resp = requests.get(
    f"{BASE_URL}/v1/events",
    params={"page": 1, "per_page": 100},
    headers=auth_headers(),
    timeout=(5, 30),           # (connect, read) seconds — ALWAYS set this
)
resp.raise_for_status()        # raises HTTPError on 4xx/5xx
payload = resp.json()
print("status:", resp.status_code)
print("keys:", list(payload.keys()))
print("rows this page:", len(payload["data"]), "| total:", payload["total"])

### Exercise A1 — Fetch and read the envelope
Call `GET /v1/events` for **page 2** with `per_page=100`. Store the parsed JSON in
`page2`, the list of records in `records2`, and the server's reported `total` in
`total_count`. Remember `raise_for_status()`.


💡 **Hint.** Same call as above with `params={"page": 2, "per_page": 100}`. The
response body has keys `data`, `page`, `per_page`, `total`, `has_more`.


In [ ]:
resp2 = None         # TODO: GET /v1/events page 2, per_page 100, with headers + timeout
# TODO: raise_for_status()
page2 = None         # TODO: resp2.json()
records2 = None      # TODO: the "data" list
total_count = None   # TODO: the "total" field

In [ ]:
check("A1: page 2 returned 100 records", lambda: len(records2) == 100)
check("A1: total is 252", lambda: total_count == 252)
check("A1: records are dicts with event_id", lambda: "event_id" in records2[0])

### Exercise A2 — Fail loudly on a missing token
Auth failures should surface immediately. Make the **same** request but with an
**empty** headers dict (no token). Capture the status code in `bad_status`, and set
`raised` to `True` if `raise_for_status()` raises an `HTTPError`.


💡 **Hint.** `requests.get(..., headers={}, timeout=...)` returns a `401`. Wrap
`resp.raise_for_status()` in `try/except requests.exceptions.HTTPError` and set
`raised = True` in the except block.


In [ ]:
resp_bad = None      # TODO: same GET but headers={} (no token)
bad_status = None    # TODO: the status code
raised = False       # TODO: set True if raise_for_status() raises HTTPError

In [ ]:
check("A2: missing token yields 401", lambda: bad_status == 401)
check("A2: raise_for_status() raised on the 401", lambda: raised is True)

## Part B — Sessions: shared auth + connection pooling

A `requests.Session` sets headers once (applied to every request), reuses TCP
connections instead of reopening a socket each call, and works as a context manager
so connections are released cleanly. Use one whenever you make more than one call.


In [ ]:
with requests.Session() as session:
    session.headers.update(auth_headers())          # set auth ONCE
    r1 = session.get(f"{BASE_URL}/v1/events", params={"page": 1, "per_page": 50}, timeout=10)
    r2 = session.get(f"{BASE_URL}/v1/events", params={"page": 2, "per_page": 50}, timeout=10)
    r1.raise_for_status(); r2.raise_for_status()
print("both OK:", r1.status_code, r2.status_code)

### Exercise B1 — One session, two endpoints
Open a `Session`, set the auth headers once, and fetch **two different** endpoints:
`GET /v1/events` (page 1, per_page 50) into `ev` and
`GET /v1/articles/nested` (page 1, per_page 20) into `arts`. Store the two status
codes in `statuses` (a list) and the number of article records in `n_articles`.


💡 **Hint.** `with requests.Session() as s: s.headers.update(auth_headers()); ...`.
Because the header is on the session, individual `.get()` calls don't repeat it.


In [ ]:
statuses = []
n_articles = None
with requests.Session() as s:
    # TODO: set auth headers on the session once
    # TODO: GET /v1/events page1 per_page50 -> ev ; append status
    # TODO: GET /v1/articles/nested page1 per_page20 -> arts ; append status
    ev = None
    arts = None

In [ ]:
check("B1: both requests returned 200", lambda: statuses == [200, 200])
check("B1: got 20 article records", lambda: n_articles == 20)

## Part C — Pagination: page / per_page

Production APIs rarely return everything at once. The **page/per_page** pattern:
request page 1, 2, 3, …; stop when a page comes back **smaller than `per_page`**
(that's the last page). Never assume one response is the whole dataset.


### Exercise C1 — Assemble the full dataset
Write `fetch_all_pages(base_url, per_page=100)` that walks `/v1/events` page by page
using a `Session` and returns the complete list of records. Stop when a page has
fewer than `per_page` items. Call it into `all_events` and record the number of
requests it made in `n_pages`.


💡 **Hint.** Loop with `page = 1`; each iteration `GET` with
`params={"page": page, "per_page": per_page}`, `raise_for_status()`, extend your
list with `payload["data"]`, and `break` when `len(batch) < per_page`. There are 252
records, so per_page=100 → pages of 100, 100, 52 → **3 requests**.


In [ ]:
def fetch_all_pages(base_url, per_page=100):
    records = []
    n_requests = 0
    # TODO: open a Session with auth headers
    # TODO: loop pages, extend records, break on a short page
    # TODO: count requests in n_requests
    return records, n_requests

all_events, n_pages = None, None

In [ ]:
check("C1: fetched all 252 events", lambda: len(all_events) == 252)
check("C1: took 3 requests (100+100+52)", lambda: n_pages == 3)
check("C1: event_ids are unique-ish dicts", lambda: all("event_id" in e for e in all_events[:5]))

## Part D — Cursor pagination

Many modern APIs use a **cursor** (a.k.a. next-token) instead of page numbers: each
response carries a `next_cursor`; you pass it back on the next request and stop when
it's `null`. Mixing this up with page/per_page is a common source of silent
truncation.


### Exercise D1 — Follow the cursor
Write `fetch_by_cursor(base_url, limit=100)` for `/v1/events/cursor`. Start with no
cursor; each response has `data` and `next_cursor`. Pass the cursor back as a
`cursor` param until it comes back `None`. Return the full list into `cursor_events`.


💡 **Hint.** Keep a `params = {"limit": limit}` dict. After each call, read
`payload["next_cursor"]`; if it's falsy, `break`; otherwise set
`params["cursor"] = next_cursor` and loop.


In [ ]:
def fetch_by_cursor(base_url, limit=100):
    records = []
    params = {"limit": limit}
    # TODO: loop: GET with params, extend records, read next_cursor, stop when null
    return records

cursor_events = None

In [ ]:
check("D1: cursor pagination fetched all 252", lambda: len(cursor_events) == 252)
check("D1: same count as page pagination", lambda: len(cursor_events) == len(all_events))

## Part E — Retries & exponential backoff

Transient failures — network blips, `503`s, `429`s — are inevitable. Configure a
`Retry` strategy on an `HTTPAdapter` mounted on the `Session`, and **every** request
gets automatic retries with exponential backoff. The `/v1/unreliable` endpoint fails
a set number of times, then succeeds — perfect for seeing recovery happen.


In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Reset the server's failure counters so this demo is repeatable.
requests.post(f"{BASE_URL}/admin/reset", timeout=5)

retry = Retry(
    total=5,
    backoff_factor=0.2,                       # waits 0s, 0.4s, 0.8s, ... between tries
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    raise_on_status=False,                    # let raise_for_status() handle the final error
)
with requests.Session() as s:
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.headers.update(auth_headers())
    r = s.get(f"{BASE_URL}/v1/unreliable", params={"key": "demo", "fail_times": 2}, timeout=10)
    r.raise_for_status()
print("recovered after retries; server attempts:", r.json()["attempts"])

### Exercise E1 — Build a resilient session
Write `build_session()` that returns a `Session` with auth headers and a `Retry`
adapter (`total=5`, `backoff_factor=0.2`, `status_forcelist=[429,500,502,503,504]`,
`allowed_methods=["GET"]`). Reset the server, then use it to fetch
`/v1/unreliable?key=e1&fail_times=3`. Store the successful response's `attempts` in
`attempts_taken` (it should be 4: three failures then success).


💡 **Hint.** Mount `HTTPAdapter(max_retries=retry)` on both `"http://"` and
`"https://"`. `fail_times=3` means the server 503s three times, then the 4th attempt
returns 200 with `attempts == 4`.


In [ ]:
def build_session():
    session = requests.Session()
    # TODO: create a Retry, mount an HTTPAdapter on http:// and https://
    # TODO: set auth headers on the session
    return session

requests.post(f"{BASE_URL}/admin/reset", timeout=5)   # reset counters
attempts_taken = None
with build_session() as s:
    resp = None   # TODO: GET /v1/unreliable?key=e1&fail_times=3
    # TODO: raise_for_status(); read attempts

In [ ]:
check("E1: build_session returns a Session", lambda: isinstance(build_session(), requests.Session))
check("E1: recovered after exactly 4 attempts (3 fails + success)",
      lambda: attempts_taken == 4)

## Part F — Rate limits: the `Retry-After` header

A `429 Too Many Requests` often includes a **`Retry-After`** header telling you how
long to wait. The `/v1/rate-limited` endpoint returns `429` with `Retry-After: 1` on
the first hit, then succeeds. (A `Retry` adapter would also handle this automatically
since `429` is in the `status_forcelist`; here you'll handle it explicitly to see the
mechanism.)


### Exercise F1 — Respect `Retry-After`
Write `get_with_rate_limit(url, **params)` that GETs `url`; if the status is `429`,
read the `Retry-After` header (seconds), `time.sleep` that long, and retry; otherwise
`raise_for_status()` and return the response. Reset the server, call it on
`/v1/rate-limited?key=f1`, and store the final `attempts` in `rl_attempts`.


💡 **Hint.** `int(resp.headers.get("Retry-After", 1))` gives the wait. Loop:
`while True: resp = requests.get(...); if resp.status_code == 429: sleep; continue;
resp.raise_for_status(); return resp`.


In [ ]:
def get_with_rate_limit(url, **params):
    while True:
        resp = requests.get(url, params=params, headers=auth_headers(), timeout=10)
        # TODO: if 429, read Retry-After, sleep, and continue
        # TODO: else raise_for_status() and return resp
        return resp

requests.post(f"{BASE_URL}/admin/reset", timeout=5)
rl_attempts = None
# TODO: call get_with_rate_limit on /v1/rate-limited?key=f1 ; read attempts

In [ ]:
check("F1: recovered after the 429 (2nd attempt)", lambda: rl_attempts == 2)

## Part G — POST with a JSON body

`GET` retrieves; `POST` submits. Send a JSON body with the **`json=`** keyword —
`requests` serializes the dict and sets `Content-Type: application/json` for you.
(Use `json=`, not `data=`.) The `/v1/summarize` endpoint mimics an LLM completion:
send text, get a summary and token counts back.


### Exercise G1 — Call the summarize endpoint
`POST` to `/v1/summarize` with body
`{"text": "Retries and backoff keep pipelines resilient. Always set a timeout.",
"model": "atlas-pro"}`. Store the response JSON in `result`, the returned `summary`
string in `summary`, and the `output_tokens` in `out_tokens`.


💡 **Hint.** `requests.post(url, json=payload, headers=auth_headers(), timeout=30)`.
The response has keys `id`, `model`, `summary`, `input_tokens`, `output_tokens`.


In [ ]:
payload = {
    "text": "Retries and backoff keep pipelines resilient. Always set a timeout.",
    "model": "atlas-pro",
}
resp_post = None     # TODO: POST /v1/summarize with json=payload
result = None        # TODO: resp_post.json()
summary = None       # TODO: result["summary"]
out_tokens = None    # TODO: result["output_tokens"]

In [ ]:
check("G1: response echoes the requested model", lambda: result["model"] == "atlas-pro")
check("G1: summary is the first sentence", lambda: summary.startswith("Retries and backoff"))
check("G1: output_tokens is a positive int", lambda: isinstance(out_tokens, int) and out_tokens > 0)

## Stretch goals *(for fast finishers)*

**S1 — Production-grade fetch.** Combine `build_session()` (Part E) with pagination
(Part C) into `fetch_all_resilient(base_url)` that fetches all `/v1/events` through a
retry-enabled session. Confirm it returns 252.

**S2 — Nested JSON → DataFrame.** Fetch `/v1/articles/nested` (page 1, per_page 20),
then flatten with `pd.json_normalize(data, sep="_")` into `arts_df`. Confirm the
nested author fields became `author_name` and `author_id` columns.


In [ ]:
# S1
def fetch_all_resilient(base_url, per_page=100):
    records, page = [], 1
    # TODO: use build_session() + the pagination loop
    return records
resilient_events = None

# S2
arts_df = None       # TODO: fetch nested articles, json_normalize with sep="_"

In [ ]:
check("S1: resilient fetch returned 252", lambda: len(resilient_events) == 252)
check("S2: nested author fields were flattened",
      lambda: {"author_name", "author_id"} <= set(arts_df.columns))

## Wrap-up — what you can now do

- Make a `GET` with params, headers, and a timeout, and fail loudly with `raise_for_status()`.
- Reuse a `Session` for shared auth and connection pooling.
- Assemble a full dataset across **page/per_page** and **cursor** pagination.
- Recover from transient failures with `Retry` + `HTTPAdapter` and respect `Retry-After`.
- `POST` a JSON body with `json=`.
- Keep secrets in environment variables, never in source.

**Next:** Lab 2 — take data like this into Jupyter for disciplined exploratory
analysis, and learn to avoid the hidden-state traps that sink reproducibility.


In [ ]:
# Shut the local API down cleanly at the end of the notebook.
server.should_exit = True
time.sleep(0.3)
print("Local API stopped.")